# Project 3 - Turkish ASR: full run (Google Colab)

Runs the whole project in order: data split, baselines, fine-tuning, final test evaluation, results.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. In Google Drive, create the folder `ASR_Project3` and put in it:
   - the Common Voice Turkish archive you downloaded (`*.tar.gz`)
   - `Project_3_ASR_Turkish.zip` (this repository) - or set `GITHUB_URL` below instead.
3. Run the cells from top to bottom. After every step the results are copied to Drive,
   so a disconnect does not lose finished work.

Rough times on a T4: setup + data 15-25 min, baselines 30-40 min, fine-tuning 30-45 min.

In [ ]:
DATASET_VERSION = "27.0"   # write the exact Common Voice release you downloaded
GITHUB_URL = ""            # e.g. "https://github.com/<user>/<repo>.git"; leave empty to use the zip in Drive
DRIVE_DIR = "/content/drive/MyDrive/ASR_Project3"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code, packages and dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess
os.chdir('/content')
if GITHUB_URL:
    !git clone -q {GITHUB_URL} Project_3_ASR_Turkish
else:
    !unzip -q -o "{DRIVE_DIR}/Project_3_ASR_Turkish.zip" -d /content
REPO = '/content/Project_3_ASR_Turkish'
os.chdir(REPO)
!ls

In [ ]:
# Colab already has a CUDA build of torch; this keeps it and installs the rest.
!pip install -q -r requirements.txt
!pip install -q qwen-asr sherpa-onnx
import transformers, torch; print(transformers.__version__, torch.__version__, torch.cuda.is_available())

In [ ]:
# Extract Common Voice to the fast local disk and find the Turkish folder (train.tsv + clips/).
archives = glob.glob(f"{DRIVE_DIR}/*.tar.gz") + glob.glob(f"{DRIVE_DIR}/*.tar") + glob.glob(f"{DRIVE_DIR}/*.tgz")
assert archives, f"No Common Voice archive found in {DRIVE_DIR}"
print("Archive:", archives[0])
os.makedirs('/content/cv', exist_ok=True)
!tar -xf "{archives[0]}" -C /content/cv
CV_DIR = os.path.dirname([p for p in glob.glob('/content/cv/**/train.tsv', recursive=True)][0])
print("Common Voice folder:", CV_DIR); print(os.listdir(CV_DIR))

In [ ]:
import subprocess
def backup():
    """Copy everything except audio and checkpoints to Drive."""
    subprocess.run(f'cd {REPO} && rm -f /content/results_backup.zip && zip -q -r /content/results_backup.zip results data/manifest.csv data/dataset_info.json report presentation -x "results/checkpoints/*"', shell=True, check=True)
    subprocess.run(f'cp /content/results_backup.zip "{DRIVE_DIR}/results_backup.zip"', shell=True, check=True)
    print("Backed up to Drive")

## 2. Part 1 (Cinar) - Data: speaker-separated splits
Uses the official Common Voice train/dev/test files and checks that no speaker is in two splits.
Each split is reduced by keeping whole speakers so everything fits in one evening: validation/test keep max 20 clips
per speaker; train keeps max 250, because the official train split has only 28 speakers.
To use more data, raise `--max-train` / `--max-test` (0 = all).

In [ ]:
!python src/prepare_common_voice.py --cv-dir "{CV_DIR}" --dataset-version {DATASET_VERSION} --split-mode official --seed 42 --max-train 6000 --max-validation 600 --max-test 2000 --max-clips-per-speaker 20 --train-max-clips-per-speaker 250 --workers 2
backup()

## 3. Part 2 (Bashir) - Baselines, no training by us
Every model is downloaded at the exact commit saved in `results/model_revisions.json`
and scored on the same test clips with the same normalization.

In [ ]:
!python src/record_model_revisions.py

In [ ]:
!python src/evaluate_whisper.py --model openai/whisper-small --tag whisper_small --batch-size 16
backup()

In [ ]:
!python src/evaluate_wav2vec2.py --model m3hrdadfi/wav2vec2-large-xlsr-turkish --tag wav2vec2_xlsr_tr --batch-size 16
backup()

In [ ]:
!python src/evaluate_qwen3.py --model Qwen/Qwen3-ASR-0.6B --tag qwen3_asr_0.6b --batch-size 16
backup()

In [ ]:
!python src/evaluate_transducer.py --model atasoglu/seda-v0.1 --tag seda_rnnt --threads 2
backup()

## 4. Part 3 (Nader) - Our contribution: fine-tuning
Trains on our train split only. Validation picks the best epoch. The test split is not touched here.
If you get a CUDA out-of-memory error, use `--batch-size 4 --grad-accum 4`.

In [ ]:
!python src/train_wav2vec2.py --model m3hrdadfi/wav2vec2-large-xlsr-turkish --epochs 3 --lr 2e-5 --batch-size 8 --grad-accum 2 --seed 42
backup()

Settings are now frozen. The fine-tuned model is scored **once** on the test split.

In [ ]:
!python src/evaluate_wav2vec2.py --model results/checkpoints/wav2vec2_finetuned/best --tag wav2vec2_xlsr_tr_finetuned --role contribution --base-model m3hrdadfi/wav2vec2-large-xlsr-turkish --batch-size 16
backup()

## 5. Results, error analysis, report and slides

In [ ]:
!python src/summarize_results.py
!python src/error_analysis.py
!python src/fill_results.py
backup()

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/RESULTS.md', encoding='utf-8').read()))

## 6. One upload package per student

Each student uploads **his own part** to the GitHub repository. This cell builds three zip files in Drive
(`ASR_Project3/upload_parts/`) with each student's code **and** his results. Unzip yours and drag the
folders into GitHub (Add file > Upload files), keeping the folder structure.

In [ ]:
import zipfile, pathlib
PARTS = {
    "Part1_Cinar_Data": ["src/prepare_common_voice.py", "data/README.md", "data/manifest.csv",
                         "data/dataset_info.json", "results/data"],
    "Part2_Bashir_Baselines": ["src/record_model_revisions.py", "src/evaluate_whisper.py", "src/evaluate_wav2vec2.py",
                               "src/evaluate_qwen3.py", "src/evaluate_transducer.py", "docs/baselines.md",
                               "results/baseline_matrix.csv", "results/model_revisions.json",
                               "results/systems/whisper_small", "results/systems/wav2vec2_xlsr_tr",
                               "results/systems/qwen3_asr_0.6b", "results/systems/seda_rnnt"],
    "Part3_Nader_Contribution": ["README.md", "requirements.txt", ".gitignore", "src/common.py",
                                 "src/train_wav2vec2.py", "src/summarize_results.py", "src/error_analysis.py",
                                 "src/fill_results.py", "notebooks", "results/training",
                                 "results/systems/wav2vec2_xlsr_tr_finetuned", "results/summary.csv",
                                 "results/comparison.json", "results/RESULTS.md", "results/error_analysis",
                                 "report", "presentation"],
}
out = pathlib.Path(DRIVE_DIR) / "upload_parts"
out.mkdir(exist_ok=True)
root = pathlib.Path(REPO)
for name, items in PARTS.items():
    with zipfile.ZipFile(out / f"{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for item in items:
            p = root / item
            files = [p] if p.is_file() else sorted(f for f in p.rglob("*") if f.is_file()) if p.exists() else []
            if not files:
                print(f"  (missing, not run yet) {item}")
            for f in files:
                if "checkpoints" not in f.parts and "__pycache__" not in f.parts:
                    z.write(f, f.relative_to(root).as_posix())
    print("Saved", out / f"{name}.zip")

Done. Results backup: `ASR_Project3/results_backup.zip`. Upload packages: `ASR_Project3/upload_parts/`.